# Lepton reconstruction efficiency vs $L_{xy}$

Measure the reconstruction efficiency of electrons, photons, PF muons, and DSA muons as a function of the gen-level dark photon $L_{xy}$.

**How the efficiency is defined.** The efficiency is measured per gen dark photon ($A$), not per gen lepton:
1. Take the gen dark photons of the right decay flavor: electrons and photons are compared to $A \to ee$, PF and DSA muons to $A \to \mu\mu$.
2. For each dark photon, count the reco objects of the given type within $\Delta R < 0.5$ of it ($n_\mathrm{near}$). This is the same count used by the `*_nearGenA_n_genA_lxy` hists in `leptonReco.ipynb`. Only reco objects that pass the channel's object cuts are counted.
3. The dark photon counts as reconstructed if $n_\mathrm{near} \geq$ `min_n`.
4. $\epsilon(L_{xy}) = \dfrac{\text{number of reconstructed dark photons in the } L_{xy} \text{ bin}}{\text{number of dark photons in the } L_{xy} \text{ bin}}$

**What `min_n` does.** `min_n` is the minimum number of associated reco objects a dark photon needs to enter the numerator:
- `min_n = 1`: at least one lepton of the pair was reconstructed, $\epsilon_{\geq 1}$
- `min_n = 2`: both leptons were reconstructed, $\epsilon_{\geq 2} \leq \epsilon_{\geq 1}$

For very collimated pairs a single reco object can lie within $\Delta R < 0.5$ while the other lepton is lost, which still passes `min_n = 1`, so `min_n = 2` is the cleaner "both reconstructed" test.

The numerator and denominator hists are defined at the end of `sidm/definitions/hists.py` and grouped in the `lepton_recoEff_lxy` hist collection. Uncertainties are 68.3% Clopper-Pearson intervals.

In [ ]:
# python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
import awkward as ak
import hist
from dask.distributed import Client, performance_report
# numerics
import numpy as np
from scipy import stats
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
importlib.reload(llpnanoaodschema)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline

In [ ]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Run over every available lifetime of four (bound state mass, dark photon mass) points that more or less span the possible gen-lepton $\Delta R$ values. Each sample is plotted on its own; samples are never summed or mixed. Hists are filled unweighted (raw counts), so the binomial uncertainty is exact. Several channels are run at once (e.g. `base` vs `base_sr`, or `baseNoLjNoLjsource` vs `baseNoLj` to see how much of the inefficiency comes from reconstruction itself vs. the LJ-source object selection) and shown side by side.

In [ ]:
# all available lifetimes for each (bound state mass, dark photon mass) point
mass_points = {
    "100GeV_0p25GeV":  ["0p02mm", "0p2mm", "2p0mm", "10p0mm", "20p0mm"],
    "100GeV_5p0GeV":   ["0p4mm", "4p0mm", "40p0mm", "200mm", "400mm"],
    "1000GeV_0p25GeV": ["0p002mm", "0p02mm", "0p2mm", "1p0mm", "2p0mm"],
    "1000GeV_5p0GeV":  ["0p04mm", "0p4mm", "4p0mm", "20p0mm", "40p0mm"],
}
# every sample is processed and plotted separately
samples = [f"2Mu2E_{mp}_{ctau}" for mp, ctaus in mass_points.items() for ctau in ctaus]

max_files = -1
fileset = utilities.make_fileset(samples, "llpNanoAOD_v2", max_files=max_files, location_cfg="signal_2mu2e_v10.yaml")

runner = processor.Runner(
    #executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
)

# channels to compare (each is plotted in its own column, never on the same axes)
#channels = ["baseNoLjNoLjsource", "baseNoLj"]  # without and with the LJ-source cuts
channels = ["base", "base_sr"]
p = sidm_processor.SidmProcessor(
    channels,
    ["lepton_recoEff_lxy"],
    unweighted_hist=True,  # raw counts -> exact binomial uncertainties
)

In [ ]:
output = runner.run(fileset, treename="Events", processor_instance=p)
out = output["out"]

In [ ]:
#Or, load already processed data
import coffea.util
output = coffea.util.load('output_leptonRecoEff_vs_lxy.coffea')
out = output["out"]

Helper functions to compute the efficiency with a Clopper-Pearson interval and to plot it vs $L_{xy}$ for all four object types.

In [ ]:
# reco object -> (plot label, denominator hist, color). The denominator is set by the gen dark
# photon decay flavor: electrons/photons use A->ee, PF/DSA muons use A->mumu.
# Colors are fixed so each object type looks the same in every panel.
reco_objs = {
    "electron": ("Electron", "genAs_toE_lxy_recoEff",  "tab:blue"),
    "photon":   ("Photon",   "genAs_toE_lxy_recoEff",  "tab:orange"),
    "muon":     ("PF muon",  "genAs_toMu_lxy_recoEff", "tab:red"),
    "dsaMuon":  ("DSA muon", "genAs_toMu_lxy_recoEff", "tab:purple"),
}


def clopper_pearson(k, n, cl=0.682689):
    # Clopper-Pearson interval for k passing out of n trials (k, n may be non-integer effective counts)
    k, n = np.asarray(k, dtype=float), np.asarray(n, dtype=float)
    alpha = 1 - cl
    with np.errstate(divide="ignore", invalid="ignore"):
        lo = np.where(k > 0, stats.beta.ppf(alpha/2, k, n - k + 1), 0.0)
        hi = np.where(k < n, stats.beta.ppf(1 - alpha/2, k + 1, n - k), 1.0)
    return lo, hi


def efficiency(num, den, cl=0.682689):
    # Per-bin efficiency num/den and its asymmetric (down, up) errors.
    #   den: Lxy of every gen dark photon (min_n = 0)
    #   num: Lxy of the gen dark photons with >= min_n associated reco objects (a subset of den)
    # Bins with an empty denominator are nan. The interval uses effective counts
    # sum(w)^2/sum(w^2), which equal the raw counts for unweighted hists and keep weighted hists
    # usable as well.
    n, k, n_var = den.values(), num.values(), den.variances()
    with np.errstate(divide="ignore", invalid="ignore"):
        eff = np.where(n > 0, k/n, np.nan)
        n_eff = np.where(n_var > 0, n**2/n_var, 0.0)
    k_eff = np.clip(eff, 0, 1)*n_eff
    lo, hi = clopper_pearson(k_eff, n_eff, cl)
    return eff, eff - lo, hi - eff


def eff_hist(num, den):
    # Return an efficiency hist (variance = symmetrized error^2) and the asymmetric errors to plot
    eff, err_down, err_up = efficiency(num, den)
    h = hist.Hist(*den.axes, storage=hist.storage.Weight())
    h.view().value[:] = eff
    h.view().variance[:] = (0.5*(err_down + err_up))**2
    return h, np.array([err_down, err_up])

In [ ]:
def plot_eff_panel(ax, sample_hists, channel, min_n, cms_label=True):
    # Efficiency of all four object types vs Lxy for ONE sample and ONE channel.
    # min_n selects the numerator: 1 = >=1 associated reco object, 2 = >=2 (both leptons).
    plt.sca(ax)
    for i, (obj, (label, den_name, color)) in enumerate(reco_objs.items()):
        num = sample_hists[f"{obj}_nearGenA_ge{min_n}_genA_lxy"][channel, :]
        den = sample_hists[den_name][channel, :]
        h_eff, errors = eff_hist(num, den)
        utilities.plot(h_eff, histtype="errorbar", yerr=np.nan_to_num(errors), flow="none",
                       skip_label=(i > 0 or not cms_label), label=label, color=color, markersize=12)
    ax.legend(title=f"{channel}\n" + rf"$\geq${min_n} reco within $\Delta R<0.5$ of dark photon",
              alignment="left", loc="best")
    return den.axes[0].edges


def plot_eff_vs_lxy(proc_output, sample, channels, logx=True):
    # One figure per sample (samples are never summed). Channels are compared side by side instead
    # of on the same axes: each column is one channel, the top row is min_n = 1, the bottom row is
    # min_n = 2. All panels share the same x and y axes, so a point in one column can be compared
    # directly with the same point in the neighboring column; horizontal grid lines help with this.
    ncol = len(channels)
    fig, axes = plt.subplots(2, ncol, figsize=(13*ncol, 20), sharex=True, sharey=True, squeeze=False)
    for col, channel in enumerate(channels):
        for row, min_n in enumerate([1, 2]):
            ax = axes[row, col]
            edges = plot_eff_panel(ax, proc_output[sample]["hists"], channel, min_n, cms_label=(row == 0))
            ax.grid(axis="y", alpha=0.4)
            ax.set_ylim(0, 1.2)
            if logx:
                ax.set_xscale("log")
                ax.set_xlim(0.3*edges[1], edges[-1])
            if row == 0:
                ax.set_xlabel(None)
            ax.set_ylabel(rf"Reco efficiency ($\geq${min_n})" if col == 0 else None)
    fig.suptitle(sample_title(sample), y=0.995)
    fig.subplots_adjust(wspace=0.04, hspace=0.04, top=0.93)
    #plt.savefig(f"{sidm_path}/plots/lepton_recoEff_vs_lxy_{sample}.png", bbox_inches="tight")
    return fig


def sample_title(sample):
    # "2Mu2E_100GeV_0p25GeV_20p0mm" -> readable title
    _, bs_mass, dp_mass, ctau = sample.split("_")
    bs_mass = bs_mass.replace("GeV", " GeV")
    dp_mass = dp_mass.replace("p", ".").replace("GeV", " GeV")
    ctau = ctau.replace("p", ".").replace("mm", " mm")
    return f"Bound state mass = {bs_mass}, dark photon mass = {dp_mass}, " + rf"$c\tau$ = {ctau}"

Efficiency vs $L_{xy}$, one figure per sample. Columns: channels; rows: $\geq 1$ and $\geq 2$ associated reco objects.

In [ ]:
for sample in samples:
    plot_eff_vs_lxy(out, sample, channels)

In [ ]:
# save output
import coffea.util
coffea.util.save(output, "output_leptonRecoEff_vs_lxy.coffea")